# BM25 Retrieval Index

This notebook refreshes the complete lexical retrieval index from the current OCR corpus. It prepares legal articles, creates chunks, tokenizes the searchable text, and stores the BM25 index in SQLite FTS5. It does not create embeddings.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data-pipeline').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'data-pipeline'))

from apps.core.retrieval_config import PipelineConfig
from bm25.build_pipeline import build_bm25_pipeline
from apps.core.bm25.sqlite_retriever import BM25Retriever

config = PipelineConfig(project_root=PROJECT_ROOT)
config.create_output_dirs()
print('Project root:', PROJECT_ROOT)
print('OCR directory:', config.ocr_dir)
print('Documents CSV:', config.documents_csv)
print('Output directory:', config.output_dir)

## 1. Refresh articles, chunks, and the BM25 database

Running this cell rebuilds the SQLite database from the current corpus. The old database is replaced only after the new database is successfully built.

In [ ]:
index_path = build_bm25_pipeline(config)
print('BM25 database:', index_path)

In [ ]:
import json

article_count = sum(1 for line in config.article_path.read_text(encoding='utf-8').splitlines() if line.strip())
chunk_count = sum(1 for line in config.chunk_path.read_text(encoding='utf-8').splitlines() if line.strip())

print('Articles:', article_count)
print('Chunks:', chunk_count)
print('BM25 database exists:', config.bm25_path.exists())
print('BM25 database size:', config.bm25_path.stat().st_size, 'bytes')

## 2. Test BM25 search

These queries confirm that the stored index can be loaded and used by the application retriever.

In [ ]:
retriever = BM25Retriever(PROJECT_ROOT, config)
retriever.load()

for query in ['ច្បាប់', 'តុលាការ', '78']:
    print(f'\nQuery: {query}')
    for result in retriever.search(query, top_k=5):
        print(result['rank'], result['article_id'], result['article_number'], round(result['bm25_score'], 3))

## 3. Evaluate BM25

The evaluation uses manually judged queries from `data/evaluation/relevance_mapping.json`. Results are matched at the canonical article level: `document_id-article_number`. Higher values mean the relevant legal article appears nearer the top.

In [ ]:
import json
from bm25.evaluate import evaluate

evaluation_path = PROJECT_ROOT / 'data' / 'evaluation' / 'relevance_mapping.json'
cases = json.loads(evaluation_path.read_text(encoding='utf-8'))
metrics = evaluate(retriever, cases, k=5)
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print('Interpretation: higher Recall@5, Hit@5, and MRR@3 are better.')

## Output files

The refreshed artifacts are written to `data/dense_retrieval/`:

- `articles.jsonl`
- `chunks.jsonl`
- `bm25.sqlite3`